In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
##### imports & global config #####
import json, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from collections import OrderedDict
import random


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

plt.rcParams["figure.dpi"] = 120
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

PROJECT_DIR   = Path("/content/drive/My Drive/Hex525_Data_Science_Project")
DATA_DIR      = PROJECT_DIR / "Data"
COHORT_DIR    = DATA_DIR / "Cohort3"
ARTIFACTS_DIR = PROJECT_DIR / "Artifacts2"
RESULTS_DIR   = PROJECT_DIR / "Results2"
PARQUET_FILE = DATA_DIR / "nsw_demand_features2.parquet"

# forecasting window (48 slots in and 48 slots out)
LOOKBACK = 48   # 24h history (30-min slots)
HORIZON  = 48   # 24h forecast

# hyperparameters
HIDDEN   = 64
LR       = 1e-3
BATCH    = 256
EPOCHS   = 80
PATIENCE = 10
CLIP_NORM = 1.0     # gradient clipping to avoid NaN

# time-based split
TRAIN_END = "2017-12-31 23:30:00"  # train: start .. 2017-12-31
VAL_END   = "2018-12-31 23:30:00"  # val:   2018-01-01 .. 2018-12-31

TARGET_COL = "y"                 # demand
HOT_DAY    = "is_hot_day"
HOT_SLOT   = "is_hot_slot"

# 2 features: demand history + temp (minimal)
fs2_min = [
        "y_lag1",          # most recent half-hour load
        "temp",            # current half-hour temperature
]

# 3 features: demand + temp + time-of-day
fs3_time = [
        "y_lag1",
        "temp",
        "sin_hh", "cos_hh",   # 48-slot daily cycle
]

# 4 features: demand + temp + time + calendar (weekend/holiday effects)
fs4_cal = [
        "y_lag1",
        "temp",
        "sin_hh", "cos_hh",
        "is_weekend", "is_holiday",
]

# full features based
fs_strong = [
        # demand history
        "y_lag1", "y_lag48", "y_lag336",
        "y_roll_mean_48", "y_roll_std_48", "y_roll_mean_336",
        # temperature
        "temp", "temp_lag1", "temp_lag2", "temp_lag3", "temp_lag48",
        "temp_roll_max_48", "temp_roll_min_48",
        # calendar/time
        "sin_hh", "cos_hh", "sin_dow", "cos_dow",
        "is_weekend", "is_holiday",
]

FEATURE_SETS = {
    "fs2_min":  fs2_min,
    "fs3_time": fs3_time,
    "fs4_cal":  fs4_cal,
    "fs_strong":fs_strong,
}

# LSTM modelling

In [ ]:
# load data
df_all = pd.read_parquet(PARQUET_FILE)
print("Loaded dataset with shape:", df_all.shape)
print(df_all.head())

if not isinstance(df_all.index, pd.DatetimeIndex):
    df_all.index = pd.to_datetime(df_all.index)
if df_all.index.tz is not None:
    df_all.index = df_all.index.tz_localize(None)

df_all = df_all.sort_index()

chosen_set   = "fs_strong"
FEATURES     = FEATURE_SETS[chosen_set]
cols_to_check = FEATURES + ["y"]

print(f"Using feature set: {chosen_set} ({len(FEATURES)} features)")
print(FEATURES)


# Drop NaN
na_counts = df_all[cols_to_check].isna().sum().sort_values(ascending=False)
print("\nTop NA counts among selected features:")
print(na_counts.head(10))

print("Rows with ANY NaN among selected features/target:",
      df_all[cols_to_check].isna().any(axis=1).sum())
print("Shape before dropna:", df_all.shape)

df_all = df_all.dropna(subset=cols_to_check).copy()
print("Shape after dropna:", df_all.shape)


# time split
idx = df_all.index
df_train = df_all.loc[idx <= TRAIN_END].copy()
df_val   = df_all.loc[(idx > TRAIN_END) & (idx <= VAL_END)].copy()
df_test  = df_all.loc[idx > VAL_END].copy()

print(f"Split sizes: train={len(df_train)}, val={len(df_val)}, test={len(df_test)}")


print(f"Total rows: {len(df_all):,}")
print(
    f"Train: {len(df_train):,} rows "
    f"({df_train.index.min().date()} -> {df_train.index.max().date()})"
)
print(
    f"Val:   {len(df_val):,} rows "
    f"({df_val.index.min().date()} -> {df_val.index.max().date()})"
)
print(
    f"Test:  {len(df_test):,} rows "
    f"({df_test.index.min().date()} -> {df_test.index.max().date()})"
)

# New Cohort 3 building

In [ ]:
# Build Cohort3 from the test split

PROJECT_DIR = Path("/content/drive/My Drive/Hex525_Data_Science_Project")
DATA_DIR     = PROJECT_DIR / "Data"
COHORT3_DIR  = DATA_DIR / "Cohort3"
COHORT3_DIR.mkdir(parents=True, exist_ok=True)

def _tz_naive(idx: pd.Index) -> pd.DatetimeIndex:
    if not isinstance(idx, pd.DatetimeIndex):
        idx = pd.to_datetime(idx)
    if idx.tz is not None:
        idx = idx.tz_convert(None) if hasattr(idx, "tz_convert") else idx.tz_localize(None)
    return idx.round("30min")

df_te = df_test.copy()

df_te.index = _tz_naive(df_te.index)
df_te = df_te.sort_index()

need_cols = {
    "y": "true",
    "y_forecast_latest": "op_latest",
    "y_forecast_24h_latest": "op_24h_latest",
}
missing = [c for c in need_cols if c not in df_te.columns]
if missing:
    raise ValueError(f"Missing required columns in df_test: {missing}")

if "is_hot_day" not in df_te.columns:
    raise ValueError("df_test is missing 'is_hot_day'. Add it during feature engineering.")

# assemble base cohort frame
cohort = pd.DataFrame(index=df_te.index)
cohort["is_hot_day"]     = df_te["is_hot_day"].astype(int)
cohort["true"]           = df_te["y"].astype(float)
cohort["op_latest"]      = df_te["y_forecast_latest"].astype(float)
cohort["op_24h"]         = df_te["y_forecast_24h"].astype(float)
cohort["op_24h_latest"]  = df_te["y_forecast_24h_latest"].astype(float)

# sanity check
keep = cohort["true"].notna() & cohort["op_latest"].notna()
dropped = len(cohort) - int(keep.sum())
cohort = cohort.loc[keep].copy()

print(f"[cohort3] rows kept: {len(cohort)} (dropped {dropped} with missing true/op_latest)")
print(f"[cohort3] 24h NaNs (strict): {cohort['op_24h'].isna().sum()}")
print(f"[cohort3] 24h_latest NaNs (expected where no ≤24h issue exists): {cohort['op_24h_latest'].isna().sum()}")
print(cohort.head())


print(f"[cohort3] enforcing full 48-slot days (timeline-based only)")

by_day = cohort.index.floor("D")

# keep days that have exactly 48 slots in the timeline
full_days = cohort.groupby(by_day).size().eq(48)

keep_mask = by_day.isin(full_days[full_days].index)

before = len(cohort)
cohort_full = cohort.loc[keep_mask].copy()
dropped = before - len(cohort_full)

print(f"[cohort3] full-day filter: kept {len(cohort_full)} rows, dropped {dropped}")
print(f"[cohort3] days kept: {full_days.sum()}  |  days dropped: {(~full_days).sum()}")

cohort = cohort_full

# split into overall vs. hot-day
cohort_overall = cohort.reset_index().rename(columns={"index": "timestamp"})
cohort_hotday  = cohort_overall.loc[cohort_overall["is_hot_day"] == 1].copy()


# save
overall_csv = COHORT3_DIR / "cohort_overall_all.csv"
hotday_csv  = COHORT3_DIR / "cohort_hotday_all.csv"
cohort_overall.to_csv(overall_csv, index=False)
cohort_hotday.to_csv(hotday_csv, index=False)

print(f"[save] overall_all -> {overall_csv} (rows={len(cohort_overall)})")
print(f"[save] hotday_all  -> {hotday_csv} (rows={len(cohort_hotday)})")

# verify 'true' equals df_all['y'] at the same timestamp
_sample_ts = cohort_overall["timestamp"].iloc[0]
same = float(df_all.loc[_tz_naive(pd.Index([_sample_ts]))[0], "y"])
print(f"[check] sample ts={_sample_ts} | true in cohort={cohort_overall.loc[0, 'true']:.2f} | y in store={same:.2f}")

# Cont. LSTM

In [ ]:
def make_supervised_sequences(
    X_df,                      # features
    y_ser,                     # target
    df_source=None,
    lookback: int = 48,
    horizon: int = 48,
    weight_mode: str = "none", # "none" | "hot_slot" | "hot_day"
    alpha_hot: float = 1.0,
):


    def row_slice_arr(arr, start, end):
        if hasattr(arr, "iloc"):
            return arr.iloc[start:end].to_numpy(np.float32)
        else:
            return arr[start:end].astype(np.float32)

    def row_slice_vec(vec, start, end):
        if hasattr(vec, "iloc"):
            return vec.iloc[start:end].to_numpy(np.float32)
        else:
            return vec[start:end].astype(np.float32)

    # length of X_df
    n = len(X_df)

    if df_source is not None:
        idx_all = df_source.index
    elif hasattr(X_df, "index"):
        idx_all = X_df.index
    else:
        idx_all = np.arange(n)

    have_hot_cols = False
    hot_slot = None
    hot_day  = None
    if df_source is not None:
        try:
            hot_slot = df_source.get(HOT_SLOT, None)
            hot_day  = df_source.get(HOT_DAY,  None)
            have_hot_cols = (hot_slot is not None) or (hot_day is not None)
        except Exception:
            have_hot_cols = False

    Xs, Ys, wts, idx_out = [], [], [], []

    # build windows
    for t in range(lookback, n - horizon + 1):
        Xs.append(row_slice_arr(X_df, t - lookback, t))          # (lookback, F)
        Ys.append(row_slice_vec(y_ser, t, t + horizon))          # (horizon,)
        idx_out.append(idx_all[t])

        # weights
        w = 1.0
        if weight_mode == "hot_slot" and have_hot_cols and hot_slot is not None:
            flag = hot_slot.iloc[t] if hasattr(hot_slot, "iloc") else hot_slot[t]
            w = alpha_hot if flag == 1 else 1.0
        elif weight_mode == "hot_day" and have_hot_cols and hot_day is not None:
            flag = hot_day.iloc[t] if hasattr(hot_day, "iloc") else hot_day[t]
            w = alpha_hot if flag == 1 else 1.0
        wts.append(w)

    return (
        np.asarray(Xs,  np.float32),  # (N, lookback, F)
        np.asarray(Ys,  np.float32),  # (N, horizon)
        np.asarray(wts, np.float32),  # (N,)
        np.asarray(idx_out),          # (N,)
    )

In [ ]:
# lstm model
class LSTMRegressor(nn.Module):
    def __init__(self, n_feats:int, hidden:int, out_horizon:int):
        super().__init__()
        self.rnn  = nn.LSTM(input_size=n_feats, hidden_size=hidden, batch_first=True)
        self.head = nn.Sequential(OrderedDict([
            ("in",  nn.Linear(hidden, hidden)),
            ("act", nn.ReLU()),
            ("out", nn.Linear(hidden, out_horizon)),
        ]))
    def forward(self, x):
        h,_ = self.rnn(x)
        h   = h[:, -1, :]
        return self.head(h)

def save_run(model, scX, scY, meta:dict, run_tag:str):
    p = ARTIFACTS_DIR / run_tag
    torch.save({"state_dict": model.state_dict(), "meta": meta}, f"{p}.pt")
    joblib.dump(scX, f"{p}_scalerX.joblib")
    joblib.dump(scY, f"{p}_scalerY.joblib")
    with open(f"{p}.json","w") as f: json.dump(meta, f, indent=2)


In [ ]:
def eval_on_dataframe(model, scX, scY, df_eval, feature_names, label, csv_prefix, do_plot=True):
    cont_cols = [c for c in feature_names if c not in ["is_weekend", "is_holiday", "is_long_weekend"]]
    Xc = pd.DataFrame(scX.transform(df_eval[cont_cols].astype(float)),      # scale X with train scalers
                      columns=cont_cols, index=df_eval.index)
    X  = df_eval[feature_names].copy()
    X[cont_cols] = Xc

    # scaled y
    y = scY.transform(df_eval[["y"]].astype(float)).ravel()

    # build 48 -> 48 windows
    Xte_seq, yte_seq, _wte_seq, _idx = make_supervised_sequences(
        X, y, df_eval, lookback=LOOKBACK, horizon=HORIZON, weight_mode="none"
    )

    with torch.no_grad():
        xb = torch.from_numpy(Xte_seq).to(DEVICE).float()
        y_sc = model.to(DEVICE)(xb).cpu().numpy()           # (N, 48) scaled

    # inverse-transform back to MW
    yhat  = scY.inverse_transform(y_sc.reshape(-1, 1)).reshape(y_sc.shape)
    ytrue = scY.inverse_transform(yte_seq.reshape(-1, 1)).reshape(yte_seq.shape)

    # compute curves + means
    curves   = _metrics_per_horizon(yhat, ytrue)
    mean_row = _metrics_mean(curves)

    # save summary
    tbl = pd.DataFrame([mean_row], index=[label]).round(2)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    tbl.to_csv(RESULTS_DIR / f"{csv_prefix}_means.csv")

    # save per-horizon table
    ph = pd.DataFrame({
        "horizon": np.arange(1, HORIZON + 1),
        "MAE_MW":  curves["MAE_MW"],
        "RMSE_MW": curves["RMSE_MW"],
        "MAPE_%":  curves["MAPE_%"],
        "Bias_MW": curves["Bias_MW"],
    })
    ph.to_csv(RESULTS_DIR / f"{csv_prefix}_per_horizon.csv", index=False)

    if do_plot:
        plot_horizon_curves({label: curves},
                            f"{label} – Horizon curves",
                            RESULTS_DIR / f"{csv_prefix}_curves.png")

    return curves, mean_row

In [ ]:
def train_in_memory(run_tag, feature_names, df_train, df_val,
                    hidden=HIDDEN, lr=LR, batch=BATCH, epochs=EPOCHS,
                    patience=PATIENCE, min_delta=1e-4,
                    weight_mode="none", alpha=1.0):


    # fit scalers on train only
    cont_cols = [c for c in feature_names if c not in ["is_weekend","is_holiday"]]
    scX = StandardScaler().fit(df_train[cont_cols].astype(float))
    scY = StandardScaler().fit(df_train[["y"]].astype(float))

    def transform(df):
        Xc = pd.DataFrame(scX.transform(df[cont_cols].astype(float)),
                          columns=cont_cols, index=df.index)
        X = df[feature_names].copy()
        X[cont_cols] = Xc
        y = scY.transform(df[["y"]].astype(float)).ravel()
        return X.to_numpy(dtype=np.float32), y.astype(np.float32), df

    Xtr, ytr, df_tr = transform(df_train)
    Xva, yva, df_va = transform(df_val)

    # build sequences
    Xtr_seq, ytr_seq, wtr_seq, _ = make_supervised_sequences(
        Xtr, ytr, lookback=LOOKBACK, horizon=HORIZON
    )
    Xva_seq, yva_seq, wva_seq, _ = make_supervised_sequences(
        Xva, yva, lookback=LOOKBACK, horizon=HORIZON
    )

    # model
    model = LSTMRegressor(
        n_feats=len(feature_names), hidden=hidden, out_horizon=HORIZON
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # train loop with early stopping
    best = float("inf"); no_improve = 0
    best_sd = None
    N = len(Xtr_seq); perm = np.arange(N)


    for ep in range(1, epochs+1):
        model.train()
        np.random.shuffle(perm)
        for s in range(0, N, batch):
            sel = perm[s:s+batch]
            xb = torch.from_numpy(Xtr_seq[sel]).to(DEVICE).float()
            yb = torch.from_numpy(ytr_seq[sel]).to(DEVICE).float()
            wb = torch.from_numpy(wtr_seq[sel]).to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            yhat = model(xb)
            loss = weighted_mse_torch(yhat, yb, wb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

        # val (unweighted MSE)
        model.eval()
        with torch.no_grad():
            xv = torch.from_numpy(Xva_seq).to(DEVICE).float()
            yv = torch.from_numpy(yva_seq).to(DEVICE).float()
            val_mse = ((model(xv)-yv)**2).mean().item()

        print(f"[{run_tag}] ep {ep:02d}  val_mse={val_mse:.6f}  best={best:.6f}  no_improve={no_improve}/{patience}")

        if best - val_mse > min_delta:
            best = val_mse; no_improve = 0
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"[{run_tag}] early stop.")
                break

    # restore best weights before returning
    if best_sd is not None:
        model.load_state_dict(best_sd)


    return model.cpu(), scX, scY, float(best)

In [ ]:
def _metrics_per_horizon(pred, true):
    e    = pred - true
    mae  = np.mean(np.abs(e), axis=0)
    rmse = np.sqrt(np.mean(e*e, axis=0))
    mape = np.mean(np.abs(e) / np.clip(true, 1e-6, None), axis=0)*100.0
    bias = np.mean(e, axis=0)

    return {
        "MAE_MW":  mae,
        "RMSE_MW": rmse,
        "MAPE_%":  mape,
        "Bias_MW": bias,
    }

def _metrics_mean(curves):
    return {k: float(np.mean(v)) for k,v in curves.items()}

def plot_horizon_curves(curves_dict, title, out_png):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    keys = ["MAE_MW","RMSE_MW","MAPE_%","Bias_MW"]
    titles = ["MAE (MW)","RMSE (MW)","MAPE (%)","Bias (MW)"]
    for ax, k, t in zip(axes.ravel(), keys, titles):
        for name, c in curves_dict.items():
            ax.plot(np.arange(1, HORIZON+1), c[k], label=name)
        ax.set_title(t); ax.set_xlabel("Horizon (slots)"); ax.grid(True)
    axes[0,0].legend()
    fig.suptitle(title)
    fig.tight_layout()
    if out_png:
        out_png = Path(out_png); out_png.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
def df_for_cohort(cohort_csv: Path, run_shift_sanity: bool = True) -> pd.DataFrame:

    # load cohort and normalise tz to tz-naive
    dfc = pd.read_csv(cohort_csv, parse_dates=["timestamp"]).sort_values("timestamp")
    if getattr(dfc["timestamp"].dt, "tz", None) is not None:
        dfc["timestamp"] = dfc["timestamp"].dt.tz_localize(None)

    # normalise the feature store index (df_all) to tz-naive DateTimeIndex
    if not isinstance(df_all.index, pd.DatetimeIndex):
        df_all.index = pd.to_datetime(df_all.index)
    if getattr(df_all.index, "tz", None) is not None:
        df_all.index = df_all.index.tz_localize(None)
    store = df_all

    # strict left join by timestamp
    ts = pd.to_datetime(dfc["timestamp"].values)
    common = ts.intersection(store.index)
    df_eval = store.loc[common].copy()

    # bring over only cohort-only columns to avoid clobbering store columns
    extra_cols = [c for c in dfc.columns if c not in {"timestamp"} and c not in store.columns]
    if extra_cols:
        df_eval = df_eval.join(dfc.set_index("timestamp")[extra_cols], how="left")

    # check
    print(f"[diag] {cohort_csv.name}: cohort rows={len(dfc)}  aligned rows={len(df_eval)}")
    print("[diag] first 5 cohort ts:", dfc["timestamp"].values[:5].tolist())
    print("[diag] first 5 feature-store ts:", store.index[:5].tolist())

    # check
    if len(df_eval) == 0:
        print("[warn] No overlapping timestamps. Check tz / date range.")

    if "true" in df_eval.columns:
        df_eval["true"] = df_eval["true"].astype(float)

    op_col = "op_latest" if "op_latest" in df_eval.columns else None
    if op_col is None:
        print("[note] No operator forecast column found (op_latest). That’s ok if we only score model.")
    df_eval = df_eval.sort_index()

    return df_eval

In [ ]:
def weighted_mse_torch(
    pred: torch.Tensor,             # (B, H)
    target: torch.Tensor,           # (B, H)
    w: torch.Tensor | None = None   # (B,) or None
) -> torch.Tensor:
    err2 = (pred - target) ** 2
    if w is None:
        return err2.mean()

    # guard + normalise
    w = torch.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
    if (w > 0).sum() == 0:
        return err2.mean()

    w = w.view(-1, 1).to(pred)
    return (err2 * w).sum() / (w.sum() * pred.shape[1])

def _filter_finite_windows(X, y, name):
    mask = np.isfinite(X).all(axis=(1,2)) & np.isfinite(y).all(axis=1)
    kept = mask.sum()
    dropped = len(mask) - kept
    if dropped:
        print(f"[{name}] dropping {dropped} / {len(mask)} windows with non-finite values.")
    return X[mask], y[mask], mask


In [ ]:
COHORTS = {
    "overall_all":  COHORT_DIR / "cohort_overall_all.csv",
    "hotday_all":   COHORT_DIR / "cohort_hotday_all.csv",
}

def train_then_eval(run_tag, feature_key, weight_mode="none", alpha=1.0, eval_test=False):
    feats = FEATURE_SETS[feature_key]

    print(f"\n=== Training {run_tag} | features={feature_key} | weight={weight_mode} | alpha={alpha} ===")

    model, scX, scY, best_val = train_in_memory(run_tag, feats, df_train, df_val,
                                      weight_mode=weight_mode, alpha=alpha)

    # artifacts
    meta = {
        "run_tag": run_tag,
        "lookback": LOOKBACK,
        "out_horizon": HORIZON,
        "n_feats": len(feats),
        "feature_names": feats,
        "hidden": HIDDEN,
        "lr":LR,
        "batch": BATCH,
        "epochs": EPOCHS,
        "best_val_mse": best_val,
        "weight_mode": weight_mode,
        "alpha_hot": float(alpha),
        "torch": torch.__version__,
    }
    save_run(model.cpu(), scX, scY, meta, run_tag)
    model.to(DEVICE)

    # regular test split
    if eval_test:
        _ = eval_on_dataframe(model, scX, scY, df_test, feats,
                          label=run_tag,
                          csv_prefix=f"{run_tag}_test")

    # cohorts
    for name, path in COHORTS.items():
        try:
            dfe = df_for_cohort(path)

            # normalise column names to lowercase
            dfe.columns = [c.lower() for c in dfe.columns]

            _ = eval_on_dataframe(model, scX, scY, dfe, feats,
                                  label=f"{run_tag} ({name})",
                                  csv_prefix=f"{run_tag}_{name}")

        except Exception as e:
            print(f"[warn] {run_tag} cohort '{name}' failed: {e}")




In [ ]:
# Four baselines for RQ1
train_then_eval("lstm_fs2_min_lb48_v1",   "fs2_min",   weight_mode="none", eval_test=False)
train_then_eval("lstm_fs3_time_lb48_v1",  "fs3_time",  weight_mode="none", eval_test=True)
train_then_eval("lstm_fs4_cal_lb48_v1",   "fs4_cal",   weight_mode="none", eval_test=True)
train_then_eval("lstm_fs_strong_lb48_v1", "fs_strong", weight_mode="none", eval_test=True)

# Hot-day weighted variants on fs_strong
for a in (1.5, 2.0, 3.0):
    train_then_eval(f"lstm_fs_strong_lb48_v1_hot_day_a{str(a).replace('.','_')}",
                    "fs_strong", weight_mode="hot_day", alpha=a, eval_test=True)

# Hot-slot weighted variants on fs_strong
for a in (1.5, 2.0, 3.0):
    train_then_eval(f"lstm_fs_strong_lb48_v1_hot_slot_a{str(a).replace('.','_')}",
                    "fs_strong", weight_mode="hot_slot", alpha=a, eval_test=True)

# Analysis Tables and Plots

In [ ]:
PROJECT_DIR   = Path("/content/drive/My Drive/Hex525_Data_Science_Project")
DATA_DIR      = PROJECT_DIR / "Data"
COHORT_DIR    = DATA_DIR / "Cohort3"
ARTIFACTS_DIR = PROJECT_DIR / "Artifacts2"
RESULTS_DIR   = PROJECT_DIR / "Results2"

METRICS = ["MAE_MW", "RMSE_MW", "MAPE_%", "Bias_MW"]
PANEL_TITLES = {
    "MAE_MW": "MAE by Horizon (MW)",
    "RMSE_MW": "RMSE by Horizon (MW)",
    "MAPE_%": "MAPE by Horizon (%)",
    "Bias_MW": "Bias by Horizon (pred − true, MW)",
}

def means_csv(run_tag: str, cohort: str) -> Path:
    return RESULTS_DIR / f"{run_tag}_{cohort}_means.csv"


def horizon_csv(run_tag: str, cohort: str) -> Path:
    return RESULTS_DIR / f"{run_tag}_{cohort}_per_horizon.csv"


# load & summarise tables
def load_means_for_runs(run_map: dict, cohort: str) -> pd.DataFrame:

    rows = []
    for feat_key, tag in run_map.items():
        f = means_csv(tag, cohort)
        df = pd.read_csv(f)

        row = df.iloc[0][METRICS].to_dict()
        row.update({"feature": feat_key})
        rows.append(row)
    out = pd.DataFrame(rows).set_index("feature")[METRICS].sort_values("MAE_MW")
    return out


def add_deltas_vs_best(df: pd.DataFrame, primary: str = "MAE_MW") -> pd.DataFrame:

    best = df.loc[df[primary].idxmin(), METRICS]
    delta = df[METRICS] - best
    delta.columns = [f"Δ{c}" for c in METRICS]
    out = pd.concat([df, delta], axis=1)
    return out


def save_table(df: pd.DataFrame, out_csv: Path):
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, float_format="%.2f")


# plot horizon curves
def plot_horizon_curves(
    run_map: dict,
    cohort: str,
    title: str,
    out_png: Path,
    metrics: list = METRICS,
    legend_title: str = "Feature set",
):

    # load horizon dfs
    data = {}
    for feat_key, tag in run_map.items():
        f = horizon_csv(tag, cohort)
        df = pd.read_csv(f)
        assert "horizon" in df.columns, f"Missing 'horizon' column in {f}"
        for m in metrics:
            assert m in df.columns, f"Missing '{m}' in {f}"
        data[feat_key] = df

    # plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    ax_map = {
        "MAE_MW": axes[0, 0],
        "RMSE_MW": axes[0, 1],
        "MAPE_%": axes[1, 0],
        "Bias_MW": axes[1, 1],
    }

    colors = plt.cm.tab10.colors
    feat_order = list(run_map.keys())

    for j, feat_key in enumerate(feat_order):
        df = data[feat_key]
        for m in metrics:
            ax = ax_map[m]
            ax.plot(df["horizon"], df[m], label=feat_key, lw=2)

    for m in metrics:
        ax = ax_map[m]
        ax.set_title(PANEL_TITLES.get(m, m))
        ax.set_xlabel("Horizon (slots ahead)")
        ax.grid(True)

    # legend
    handles, labels = ax_map[metrics[0]].get_legend_handles_labels()
    fig.legend(handles, labels, title=legend_title, loc="center right")

    fig.suptitle(title, fontsize=14)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()


# baseline report
def baseline_report(
    run_map: dict,
    cohort: str,
    table_csv_name: str,
    plot_png_name: str,
    title: str,
):

    # table
    base = load_means_for_runs(run_map, cohort)
    base_d = add_deltas_vs_best(base)
    table_path = RESULTS_DIR / table_csv_name
    save_table(base_d.reset_index(), table_path)

    # plot
    plot_path = RESULTS_DIR / plot_png_name
    plot_horizon_curves(
        run_map=run_map,
        cohort=cohort,
        title=title,
        out_png=plot_path,
    )
    return base_d

In [ ]:
BASELINE_RUNS = {
    "fs2_min":  "lstm_fs2_min_lb48_v1",
    "fs3_time": "lstm_fs3_time_lb48_v1",
    "fs4_cal":  "lstm_fs4_cal_lb48_v1",
    "fs_strong":"lstm_fs_strong_lb48_v1",
}

cohort = "hotday_all"  # choose: 'overall_all' or 'hotday_all'

baseline_table = baseline_report(
    run_map=BASELINE_RUNS,
    cohort=cohort,
    table_csv_name=f"baseline_{cohort}_summary.csv",
    plot_png_name=f"baseline_{cohort}_horizon_curves.png",
    title=f"Baseline Horizon Curves — {cohort}",
)

baseline_table.style.format(precision=2)

In [ ]:
HOT_DAY_RUNS = {
    "fs_strong":        "lstm_fs_strong_lb48_v1",
    "hot_day_a1_5":     "lstm_fs_strong_lb48_v1_hot_day_a1_5",
    "hot_day_a2_0":     "lstm_fs_strong_lb48_v1_hot_day_a2_0",
    "hot_day_a3_0":     "lstm_fs_strong_lb48_v1_hot_day_a3_0",
}


# on the overall behavior
overall_hotday_tbl = baseline_report(
    HOT_DAY_RUNS, "test",
    "hotday_overall_all_summary.csv",
    "hotday_overall_all_horizon_curves.png",
    "Hot-day Weighted — overall_all",
)

# on the hot-day cohort
hotday_tbl = baseline_report(
    HOT_DAY_RUNS, "hotday_all",
    "hotday_hotday_all_summary.csv",
    "hotday_hotday_all_horizon_curves.png",
    "Hot-day Weighted — hotday_all",
)

In [ ]:
def merge_overall_hotday(overall_csv, hotday_csv, out_csv):

    # load both summary tables
    overall = pd.read_csv(overall_csv)
    hotday = pd.read_csv(hotday_csv)

    # add suffix to distinguish cohorts
    overall = overall.add_suffix("_overall")
    hotday = hotday.add_suffix("_hotday")

    # restore 'feature' column name
    overall = overall.rename(columns={"feature_overall": "feature"})
    hotday = hotday.rename(columns={"feature_hotday": "feature"})

    # merge on feature
    merged = pd.merge(overall, hotday, on="feature", how="outer")

    # save merged table
    merged.to_csv(out_csv, index=False)
    print(f"[save] merged results → {out_csv}")
    return merged


RESULTS_DIR = Path("/content/drive/My Drive/Hex525_Data_Science_Project/Results2")

merged_tbl = merge_overall_hotday(
    RESULTS_DIR / "hotday_overall_all_summary.csv",
    RESULTS_DIR / "hotday_hotday_all_summary.csv",
    RESULTS_DIR / "hotday_merged_summary.csv"
)

import IPython.display as disp
disp.display(merged_tbl)

In [ ]:

HOT_SLOT_RUNS = {
    "fs_strong":        "lstm_fs_strong_lb48_v1",
    "hot_slot_a1_5":    "lstm_fs_strong_lb48_v1_hot_slot_a1_5",
    "hot_slot_a2_0":    "lstm_fs_strong_lb48_v1_hot_slot_a2_0",
    "hot_slot_a3_0":    "lstm_fs_strong_lb48_v1_hot_slot_a3_0",
}

# on overall cohort
overall_hotslot_tbl = baseline_report(
    HOT_SLOT_RUNS, "test",
    "hotslot_overall_all_summary.csv",
    "hotslot_overall_all_horizon_curves.png",
    "Hot-slot Weighted — overall_all",
)

# on hot-day cohort
hotslot_tbl = baseline_report(
    HOT_SLOT_RUNS, "hotday_all",
    "hotslot_hotday_all_summary.csv",
    "hotslot_hotday_all_horizon_curves.png",
    "Hot-slot Weighted — hotday_all",
)

# Analysis tables and plots for final LSTM models

In [ ]:
PROJECT_DIR  = Path("/content/drive/My Drive/Hex525_Data_Science_Project")
RESULTS_DIR  = PROJECT_DIR / "Results2"
OUTPUTS_DIR  = RESULTS_DIR / "FinalOutputs"

# Overall
FS_OVERALL      = RESULTS_DIR / "lstm_fs_strong_lb48_v1_overall_all_per_horizon.csv"
HOTDAY_OVERALL  = RESULTS_DIR / "lstm_fs_strong_lb48_v1_hot_day_a1_5_overall_all_per_horizon.csv"
HOTSlot_OVERALL = RESULTS_DIR / "lstm_fs_strong_lb48_v1_hot_slot_a2_0_overall_all_per_horizon.csv"

# Hot-day cohort
FS_HOT      = RESULTS_DIR / "lstm_fs_strong_lb48_v1_hotday_all_per_horizon.csv"
HOTDAY_HOT  = RESULTS_DIR / "lstm_fs_strong_lb48_v1_hot_day_a1_5_hotday_all_per_horizon.csv"
HOTSlot_HOT = RESULTS_DIR / "lstm_fs_strong_lb48_v1_hot_slot_a2_0_hotday_all_per_horizon.csv"

def _normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    lower = {c: c.lower() for c in df.columns}
    ren = {}
    # horizon
    for c,l in lower.items():
        if l in {"horizon","slot","k","lead","step","h"}: ren[c]="horizon"; break
    # metrics
    for c,l in lower.items():
        if l in {"mae_mw","mae"}: ren[c]="MAE_MW"
    for c,l in lower.items():
        if l in {"rmse_mw","rmse"}: ren[c]="RMSE_MW"
    for c,l in lower.items():
        if l in {"mape_%","mape","mape_percent"}: ren[c]="MAPE_%"
    for c,l in lower.items():
        if l in {"bias_mw","bias","pred_true_mw","mean_bias"}: ren[c]="Bias_MW"
    df = df.rename(columns=ren)
    need = {"horizon","MAE_MW","RMSE_MW","MAPE_%","Bias_MW"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return df[list(need)].copy()

def load_one(path: Path, feature_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input not found: {path}")
    df = pd.read_csv(path)
    df = _normalize_cols(df)
    df["feature"] = feature_name
    return df

def plot_three(df: pd.DataFrame, title: str, outpath: Path):
    order = ["fs_strong", "hot_day_a1_5", "hot_slot_a2_0"]
    df["feature"] = pd.Categorical(df["feature"], categories=order, ordered=True)
    df = df.sort_values(["feature","horizon"])

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    panels = [
        ("MAE_MW", "MAE by Horizon (MW)"),
        ("RMSE_MW","RMSE by Horizon (MW)"),
        ("MAPE_%","MAPE by Horizon (%)"),
        ("Bias_MW","Bias by Horizon (pred − true, MW)")
    ]
    for ax, (col, ylabel) in zip(axes, panels):
        for feat in order:
            sdf = df[df["feature"]==feat]
            if not sdf.empty:
                ax.plot(sdf["horizon"], sdf[col], label=feat)
        ax.set_xlabel("Horizon (slots ahead)")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        if col == "RMSE_MW":
            ax.legend(title="Feature set", fontsize=9)

    fig.suptitle(title)
    fig.tight_layout()
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=200)
    plt.show()
    print(f"Saved plot → {outpath}")

print("Checking input files...")
for p in [FS_OVERALL, HOTDAY_OVERALL, HOTSlot_OVERALL, FS_HOT, HOTDAY_HOT, HOTSlot_HOT]:
    print(("✔" if p.exists() else "✘"), p)

# Overall cohort
df_overall = pd.concat([
    load_one(FS_OVERALL,      "fs_strong"),
    load_one(HOTDAY_OVERALL,  "hot_day_a1_5"),
    load_one(HOTSlot_OVERALL, "hot_slot_a2_0"),
], ignore_index=True)
plot_three(df_overall, "Final Model Comparison — Overall (all days)",
           OUTPUTS_DIR / "final_showdown_overall.png")

# Hot-day cohort
df_hotday = pd.concat([
    load_one(FS_HOT,      "fs_strong"),
    load_one(HOTDAY_HOT,  "hot_day_a1_5"),
    load_one(HOTSlot_HOT, "hot_slot_a2_0"),
], ignore_index=True)
plot_three(df_hotday, "Final Model Comparison — Hot-day cohort",
           OUTPUTS_DIR / "final_showdown_hotday.png")

break

# Check y and true value from feature engineered data and Cohort File

In [ ]:
DATA_DIR = "/content/drive/My Drive/Hex525_Data_Science_Project/Data"
orig_parquet = f"{DATA_DIR}/nsw_demand_features.parquet"

df_orig = pd.read_parquet(orig_parquet)

print("Columns:", df_orig.columns.tolist())
print(df_orig.head(5))

In [ ]:
DATA_DIR = Path("/content/drive/My Drive/Hex525_Data_Science_Project/Data")
orig_parquet = DATA_DIR / "nsw_demand_features.parquet"      # original dataset (parquet)
cohort_csv   = DATA_DIR / "Cohort2" / "cohort_overall_all.csv"  # cohort (csv)

def to_utc_naive_index(idx_like):
    idx = pd.to_datetime(idx_like)
    if getattr(idx, "tz", None) is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")
    return idx.tz_localize(None)

df_orig = pd.read_parquet(orig_parquet)
df_cohort = pd.read_csv(cohort_csv, parse_dates=["timestamp"]).set_index("timestamp")

df_orig.index   = to_utc_naive_index(df_orig.index)
df_cohort.index = to_utc_naive_index(df_cohort.index)

ts = df_cohort.index[0]

print("Sample timestamp:", ts)
print("Original y @ same ts:",
      df_orig.loc[ts, "y"] if ts in df_orig.index else "NA")
print("Cohort TRUE @ ts   :", df_cohort.loc[ts, "true"])

ts_plus_24 = ts + pd.Timedelta(hours=24)
print("Original y @ ts+24h:",
      df_orig.loc[ts_plus_24, "y"] if ts_plus_24 in df_orig.index else "NA")

tmp = df_cohort[["true"]].copy()
tmp["y_same"] = df_orig.reindex(tmp.index)["y"]
tmp["y_24h"]  = df_orig.reindex(tmp.index + pd.Timedelta(hours=24))["y"]

same_mae = (tmp["true"] - tmp["y_same"]).abs().median(skipna=True)
h24_mae  = (tmp["true"] - tmp["y_24h"]).abs().median(skipna=True)

print("\n[alignment check] median |TRUE - y_same| :", round(float(same_mae), 3))
print("[alignment check] median |TRUE - y_24h|  :", round(float(h24_mae), 3))
if h24_mae < same_mae:
    print("-> Cohort TRUE matches the original y at +24h (48 slots ahead).")
else:
    print("-> Cohort TRUE matches the original y at the same timestamp.")

In [ ]:

DATA_DIR = Path("/content/drive/My Drive/Hex525_Data_Science_Project/Data")
orig_parquet = DATA_DIR / "nsw_demand_features.parquet"
cohort_csv   = DATA_DIR / "Cohort" / "cohort_overall_all.csv"

df_orig = pd.read_parquet(orig_parquet)
df_cohort = pd.read_csv(cohort_csv, parse_dates=["timestamp"]).set_index("timestamp")

# normalise both indices
df_orig.index = pd.to_datetime(df_orig.index).tz_localize(None)
df_cohort.index = pd.to_datetime(df_cohort.index).tz_localize(None)

ts = df_cohort.index[0]

print("Sample timestamp:", ts)
print("Original y @ ts     :", df_orig.loc[ts, "y"])
print("Original y @ ts+24h :", df_orig.loc[ts + pd.Timedelta(hours=24), "y"])

# Check Cohort File

In [ ]:
import pandas as pd

cohort = pd.read_csv(COHORT_DIR / "cohort_overall_all.csv")

print("Total rows:", len(cohort))
print("NaN count in op_latest:", cohort["op_latest"].isna().sum())

# ensure timestamp is datetime
cohort["timestamp"] = pd.to_datetime(cohort["timestamp"])

# group by day (floor to daily resolution)
by_day = cohort.groupby(cohort["timestamp"].dt.floor("D")).size()

print("Days with full 48 slots:", (by_day == 48).sum(), "out of", len(by_day))
print(by_day.head())

In [ ]:
import pandas as pd

def drop_incomplete_days(in_csv: str, out_csv: str, slots_per_day: int = 48):
    """
    Drop any days that don't have the full number of slots (default = 48).
    Saves a cleaned cohort CSV.
    """
    df = pd.read_csv(in_csv, parse_dates=["timestamp"])

    # group by day and count slots
    counts = df.groupby(df["timestamp"].dt.floor("D")).size()
    full_days = counts[counts == slots_per_day].index

    # keep only rows belonging to full 48-slot days
    cleaned = df[df["timestamp"].dt.floor("D").isin(full_days)].copy()

    print(f"[clean] {in_csv}: {len(df)} rows -> {len(cleaned)} rows")
    print(f"  dropped {len(df) - len(cleaned)} rows "
          f"({len(counts) - len(full_days)} incomplete days)")

    cleaned.to_csv(out_csv, index=False)
    print(f"[save] {out_csv}")

In [ ]:
drop_incomplete_days(
    in_csv=COHORT_DIR / "cohort_overall_all.csv",
    out_csv=COHORT_DIR / "cohort_overall_all_clean.csv"
)